In [1]:
! pip install -U pypdf langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers transformers accelerate torch
! pip install -U langgraph gradio requests pydantic

  Using cached langchain-1.2.2-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_huggingface-1.2.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached langchain_core-1.2.6-py3-none-any.whl.metadata (3.7 kB)
  Using cached langgraph-1.0.5-py3-none-any.whl.metadata (7.4 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langgraph_prebuilt-1.0.5-py3-none-any.whl.metadata (5.2 kB)
Using cached langchain-1.2.2-py3-none-any.whl (105 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_text_splitters-1.1.0-py3-none-any.whl (34 kB)
Using cached langchain_huggingface-1.2.0-py3-none-any.whl (30 kB)
Usin


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached gradio-6.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.128.0-py3-none-any.whl.metadata (30 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-2.0.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.1.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.21-py3-none-any.whl.metadata (1.8 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-0.50.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tomlkit-0.13.3-py3-none-any.wh


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json
import time
import math
import requests
from typing import List, Dict, Any, TypedDict
from pydantic import BaseModel, Field, ValidationError
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langgraph.graph import StateGraph, END
import gradio as gr


c:\Users\kpaps\Desktop\CAP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Task 2: RAG Functions – Knowledge Ingestion & Retrieval

In [3]:
def extract_pdf_pages(path: str) -> List[Document]:
    reader = PdfReader(path)
    docs = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            docs.append(Document(page_content=text, metadata={"source": os.path.basename(path), "page": i}))
    return docs


def initialize_vector_store(
    pdf_path: str,
    faiss_directory: str = "./faiss_store",
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
    chunk_size: int = 500,
    chunk_overlap: int = 100,
) -> FAISS:
    os.makedirs(faiss_directory, exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

    # Load existing FAISS index
    if os.path.exists(os.path.join(faiss_directory, "index.faiss")):
        vs = FAISS.load_local(faiss_directory, embeddings, allow_dangerous_deserialization=True)
        return vs

    # Build new
    docs = extract_pdf_pages(pdf_path)
    if not docs:
        raise ValueError("No text extracted from PDF. If it is scanned (image-only), you need OCR.")

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(docs)

    vs = FAISS.from_documents(chunks, embeddings)
    vs.save_local(faiss_directory)
    return vs


def retrieve_rag_chunks(vector_store: FAISS, query: str, k: int = 3):
    retrieved_docs = vector_store.similarity_search(query, k=k)

    context = "\n\n".join(
        f"(source={d.metadata.get('source')}, page={d.metadata.get('page')})\n{d.page_content}"
        for d in retrieved_docs
    )

    citations = [{"source": d.metadata.get("source"), "page": d.metadata.get("page")} for d in retrieved_docs]
    return context, citations


def create_local_llm(model_id: str = "google/flan-t5-base", max_new_tokens: int = 220):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    gen_pipe = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens
    )
    return HuggingFacePipeline(pipeline=gen_pipe)


def generate_answer_from_context(llm: HuggingFacePipeline, context: str, query: str) -> str:
    prompt = f"""
Answer the question strictly using the context below.
If the answer is not present in the context, say: "I don't know from the provided documents."

Context:
{context}

Question: {query}
Answer:
""".strip()
    return llm.invoke(prompt)


### Task 2: Build Vector Store & Local LLM

In [4]:
pdf_path = r"/content/artificial_intelligence_tutorial.pdf"
faiss_directory = "./faiss_store"

vector_store = initialize_vector_store(pdf_path=pdf_path, faiss_directory=faiss_directory)
llm = create_local_llm(model_id="google/flan-t5-base", max_new_tokens=220)

print("✅ RAG vector_store ready + Local LLM ready")


Device set to use cpu


✅ RAG vector_store ready + Local LLM ready


### Task 4: Weather Tool Implementation

In [5]:
class WeatherToolInput(BaseModel):
    location: str = Field(..., description="City name like Chennai, Mumbai, London")
    days: int = Field(3, ge=1, le=7, description="Forecast days (1 to 7)")

def weather_tool_call(data: Dict[str, Any]) -> Dict[str, Any]:
    try:
        inp = WeatherToolInput(**data)
    except ValidationError as e:
        return {"ok": False, "error": str(e)}

    loc = inp.location.strip()
    if not loc:
        return {"ok": False, "error": "Location cannot be empty."}

    # 1) Geocode
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    try:
        gr = requests.get(geo_url, params={"name": loc, "count": 1}, timeout=15)
        gr.raise_for_status()
        gj = gr.json()
    except Exception as e:
        return {"ok": False, "error": f"Geocoding failed: {e}"}

    if not gj.get("results"):
        return {"ok": False, "error": f"Location not found: {loc}"}

    place = gj["results"][0]
    lat, lon = place["latitude"], place["longitude"]

    # 2) Forecast
    forecast_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
        "forecast_days": inp.days,
        "timezone": "auto",
    }

    try:
        fr = requests.get(forecast_url, params=params, timeout=20)
        fr.raise_for_status()
        fj = fr.json()
    except Exception as e:
        return {"ok": False, "error": f"Forecast failed: {e}"}

    daily = fj.get("daily", {})
    out = []
    n = len(daily.get("time", []))
    for i in range(n):
        out.append({
            "date": daily["time"][i],
            "temp_max_c": daily["temperature_2m_max"][i],
            "temp_min_c": daily["temperature_2m_min"][i],
            "precip_mm": daily["precipitation_sum"][i],
            "wind_max_kmh": daily["wind_speed_10m_max"][i],
        })

    return {
        "ok": True,
        "location": f"{place.get('name')}, {place.get('country')}",
        "forecast_days": inp.days,
        "forecast": out
    }


### Task 4: Calculator Tool Implementation

In [6]:
class CalculationInput(BaseModel):
    expression: str = Field(
        ..., description="Math expression, e.g. sin(pi/2) + sqrt(16) - 3**2"
    )

def calculator_tool_call(data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Evaluate an arithmetic or scientific expression using a restricted set of
    operators and functions. Supported operations include +, -, *, /, %, **,
    and functions such as sqrt, log, sin, cos, tan, exp, factorial, as well as
    the constants pi and e. Returns {'ok': True, 'result': value} on success,
    otherwise {'ok': False, 'error': ...}.
    """
    try:
        inp = CalculationInput(**data)
    except ValidationError as e:
        return {"ok": False, "error": str(e)}

    expr = inp.expression.strip()
    if not expr:
        return {"ok": False, "error": "Expression cannot be empty."}

    # Define allowed names mapping to math functions and constants
    allowed_names: Dict[str, Any] = {
        "abs": abs,
        "pow": pow,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "asin": math.asin,
        "acos": math.acos,
        "atan": math.atan,
        "sqrt": math.sqrt,
        "log": math.log,       # natural log
        "log10": math.log10,   # base-10 log
        "exp": math.exp,
        "factorial": math.factorial,
        "pi": math.pi,
        "e": math.e,
    }

    try:
        # Evaluate the expression safely with no builtins and only allowed names
        result = eval(expr, {"__builtins__": {}}, allowed_names)
        return {"ok": True, "result": result}
    except Exception as e:
        return {"ok": False, "error": f"Calculation failed: {e}"}


### Task 4: Python Custom Function Tool (10 Operations)


In [7]:
# ================================
# Task 4: Custom Python Function Tool (10 Ops)
# ================================

class PythonCustomToolInput(BaseModel):
    operation: str = Field(
        ...,
        description=(
            "Allowed operations: "
            "word_count | extract_numbers | title_case | reverse_text | "
            "lower_case | upper_case | remove_punctuation | "
            "remove_extra_spaces | sentence_count | unique_words"
        ),
    )
    text: str = Field(..., description="Input text to process")


def python_custom_tool_call(data: Dict[str, Any]) -> Dict[str, Any]:
    """Safe custom Python tool with 10 whitelisted operations only."""
    try:
        inp = PythonCustomToolInput(**data)
    except ValidationError as e:
        return {"ok": False, "error": str(e)}

    operation = inp.operation.lower().strip()
    text = inp.text or ""

    try:
        if operation == "word_count":
            words = re.findall(r"\b\w+\b", text)
            return {
                "ok": True,
                "operation": operation,
                "result": {
                    "word_count": len(words),
                    "character_count": len(text),
                    "character_count_no_spaces": len(text.replace(" ", "")),
                },
            }

        elif operation == "extract_numbers":
            numbers = re.findall(r"[-+]?\d*\.?\d+", text)
            return {"ok": True, "operation": operation, "result": numbers}

        elif operation == "title_case":
            return {"ok": True, "operation": operation, "result": text.title()}

        elif operation == "reverse_text":
            return {"ok": True, "operation": operation, "result": text[::-1]}

        elif operation == "lower_case":
            return {"ok": True, "operation": operation, "result": text.lower()}

        elif operation == "upper_case":
            return {"ok": True, "operation": operation, "result": text.upper()}

        elif operation == "remove_punctuation":
            cleaned = re.sub(r"[^\w\s]", "", text)
            return {"ok": True, "operation": operation, "result": cleaned}

        elif operation == "remove_extra_spaces":
            cleaned = re.sub(r"\s+", " ", text).strip()
            return {"ok": True, "operation": operation, "result": cleaned}

        elif operation == "sentence_count":
            endings = re.findall(r"[.!?]+", text)
            return {"ok": True, "operation": operation, "result": len(endings)}

        elif operation == "unique_words":
            words = re.findall(r"\b\w+\b", text.lower())
            return {"ok": True, "operation": operation, "result": sorted(list(set(words)))}

        else:
            return {"ok": False, "error": f"Unsupported operation: {operation}"}

    except Exception as e:
        return {"ok": False, "error": str(e)}



### Task 4: Tool Registry & Execution

### Task 1: Workflow State Definition

In [8]:
class WorkflowState(TypedDict):
    user_query: str
    operation: str            # "rag" | "tool"
    plan: str
    react_steps: List[Dict[str, Any]]

    retrieved_context: str
    citations: List[Dict[str, Any]]

    tool_name: str
    tool_input: Dict[str, Any]
    tool_result: Dict[str, Any]

    final_answer: str
    error: str


### Task 3: Planner Agent – Multi-Agent ReAct Reasoning

In [10]:
# def planning_agent(state: WorkflowState) -> WorkflowState:
#     q = state["user_query"].strip()
#     ql = q.lower()

#     # default
#     state["react_steps"] = []
#     state["error"] = ""
#     state["retrieved_context"] = ""
#     state["citations"] = []
#     state["tool_result"] = {}
#     state["tool_input"] = {}
#     state["tool_name"] = ""

#     # --- Routing rules ---
#     if "weather" in ql or "temperature" in ql or "forecast" in ql:
#         state["operation"] = "tool"
#         state["tool_name"] = "weather"
#         # Basic extraction: take words after "in"
#         # Example: "weather in chennai"
#         loc = q
#         if " in " in ql:
#             loc = q.split(" in ", 1)[1].strip()
#         else:
#             loc = "Chennai"  # fallback
#         state["tool_input"] = {"location": loc, "days": 3}
#         state["plan"] = "Call weather tool (no API key) using Open-Meteo."

#     elif "calculate" in ql or re.search(r"\d+\s*[\+\-\*\/]\s*\d+", q):
#         state["operation"] = "tool"
#         state["tool_name"] = "calculator"
#         expr = re.sub(r"(?i)\bcalculate\b", "", q).strip()
#         state["tool_input"] = {"expression": expr if expr else q}
#         state["plan"] = "Call calculator tool to compute the expression."

# elif any(k in ql for k in [
#     "word count", "count words",
#     "extract numbers",
#     "title case",
#     "reverse text",
#     "lower case",
#     "upper case",
#     "remove punctuation",
#     "remove extra spaces",
#     "sentence count",
#     "unique words",
# ]):
#     state["operation"] = "tool"
#     state["tool_name"] = "python_custom"

#     if "word count" in ql or "count words" in ql:
#         op = "word_count"
#     elif "extract numbers" in ql:
#         op = "extract_numbers"
#     elif "title case" in ql:
#         op = "title_case"
#     elif "reverse text" in ql:
#         op = "reverse_text"
#     elif "lower case" in ql:
#         op = "lower_case"
#     elif "upper case" in ql:
#         op = "upper_case"
#     elif "remove punctuation" in ql:
#         op = "remove_punctuation"
#     elif "remove extra spaces" in ql:
#         op = "remove_extra_spaces"
#     elif "sentence count" in ql:
#         op = "sentence_count"
#     elif "unique words" in ql:
#         op = "unique_words"
#     else:
#         op = "word_count"

#     text_part = q.split(":", 1)[1].strip() if ":" in q else q
#     state["tool_input"] = {"operation": op, "text": text_part}
#     state["plan"] = f"Call python_custom tool for '{op}'."

#     else:
#         state["operation"] = "rag"
#         state["plan"] = "Use RAG to retrieve context from PDF and answer grounded with citations."

#     state["react_steps"].append({"reason": state["plan"]})
#     return state

import re

def planning_agent(state: dict) -> dict:
    q = state["user_query"].strip()
    ql = q.lower()

    # default
    state["react_steps"] = []
    state["error"] = ""
    state["retrieved_context"] = ""
    state["citations"] = []
    state["tool_result"] = {}
    state["tool_input"] = {}
    state["tool_name"] = ""

    # --- Routing rules ---
    if "weather" in ql or "temperature" in ql or "forecast" in ql:
        state["operation"] = "tool"
        state["tool_name"] = "weather"
        # Basic extraction: take words after "in"
        # Example: "weather in chennai"
        if " in " in ql:
            loc = q.split(" in ", 1)[1].strip()
        else:
            loc = "Chennai"  # fallback
        state["tool_input"] = {"location": loc, "days": 3}
        state["plan"] = "Call weather tool (no API key) using Open-Meteo."

    elif "calculate" in ql or re.search(r"\d+\s*[\+\-\*\/]\s*\d+", q):
        state["operation"] = "tool"
        state["tool_name"] = "calculator"
        expr = re.sub(r"(?i)\bcalculate\b", "", q).strip()
        state["tool_input"] = {"expression": expr if expr else q}
        state["plan"] = "Call calculator tool to compute the expression."

    elif any(k in ql for k in [
        "word count", "count words",
        "extract numbers",
        "title case",
        "reverse text",
        "lower case",
        "upper case",
        "remove punctuation",
        "remove extra spaces",
        "sentence count",
        "unique words",
    ]):
        state["operation"] = "tool"
        state["tool_name"] = "python_custom"

        if "word count" in ql or "count words" in ql:
            op = "word_count"
        elif "extract numbers" in ql:
            op = "extract_numbers"
        elif "title case" in ql:
            op = "title_case"
        elif "reverse text" in ql:
            op = "reverse_text"
        elif "lower case" in ql:
            op = "lower_case"
        elif "upper case" in ql:
            op = "upper_case"
        elif "remove punctuation" in ql:
            op = "remove_punctuation"
        elif "remove extra spaces" in ql:
            op = "remove_extra_spaces"
        elif "sentence count" in ql:
            op = "sentence_count"
        elif "unique words" in ql:
            op = "unique_words"
        else:
            op = "word_count"

        text_part = q.split(":", 1)[1].strip() if ":" in q else q
        state["tool_input"] = {"operation": op, "text": text_part}
        state["plan"] = f"Call python_custom tool for '{op}'."

    else:
        state["operation"] = "rag"
        state["plan"] = "Use RAG to retrieve context from PDF and answer grounded with citations."

    state["react_steps"].append({"reason": state["plan"]})
    return state

### Task 2 & 3: RAG Agent for Retrieval

In [11]:
def retrieval_agent(state: WorkflowState) -> WorkflowState:
    q = state["user_query"]

    state["react_steps"].append({"act": "RAG.retrieve", "input": q})
    context, citations = retrieve_rag_chunks(vector_store, q, k=3)

    state["retrieved_context"] = context
    state["citations"] = citations

    state["react_steps"].append({"observe": f"Retrieved {len(citations)} chunks"})
    return state


### Task 4: Tool Agent – Multi-Tool Execution

In [12]:
def tool_execution_agent(state: WorkflowState) -> WorkflowState:
    tname = state.get("tool_name", "")
    tinp = state.get("tool_input", {})

    state["react_steps"].append({"act": "Tool.call", "tool": tname, "input": tinp})
    # dispatch to appropriate tool based on name
    if tname == "weather":
        result = weather_tool_call(tinp)
    elif tname == "calculator":
        result = calculator_tool_call(tinp)
    elif tname == "python_custom":
        result = python_custom_tool_call(tinp)
    else:
        result = {"ok": False, "error": f"Unknown tool: {tname}"}
    state["tool_result"] = result
    state["react_steps"].append({"observe": result})

    return state


### Task 3: Synthesizer Agent – Final Answer

In [13]:
def synthesis_agent(state: WorkflowState) -> WorkflowState:
    op = state.get("operation", "")

    if op == "rag":
        q = state["user_query"]
        context = state.get("retrieved_context", "")

        state["react_steps"].append({"act": "LLM.generate_grounded_answer"})
        ans = generate_answer_from_context(llm, context, q)
        state["final_answer"] = ans

        # keep citations from retrieved docs (already in state)

    elif op == "tool":
        tname = state.get("tool_name", "")
        result = state.get("tool_result", {})

        # Tool output formatting
        if tname == "weather" and result.get("ok"):
            lines = [f"Weather Forecast for {result.get('location')}:"]
            for d in result.get("forecast", []):
                lines.append(
                    f"- {d['date']}: min {d['temp_min_c']}°C, max {d['temp_max_c']}°C, "
                    f"rain {d['precip_mm']}mm, wind {d['wind_max_kmh']}km/h"
                )
            state["final_answer"] = "\n".join(lines)

        elif tname == "calculator" and result.get("ok"):
            state["final_answer"] = f"Result: {result.get('result')}"

        elif tname == "python_custom" and result.get("ok"):
            state["final_answer"] = "Python Custom Tool Output:\n" + json.dumps(result, indent=2)

        else:
            state["final_answer"] = f"Tool '{tname}' failed: {json.dumps(result, indent=2)}"

        # Tools don't produce PDF citations; keep citations empty
        state["citations"] = []

    else:
        state["final_answer"] = "Unsupported operation."
        state["citations"] = []

    return state


### Task 5: LangGraph Orchestration

In [14]:
def route_agent(state: WorkflowState) -> str:
    return "tool" if state.get("operation") == "tool" else "rag"

graph = StateGraph(WorkflowState)

graph.add_node("planner", planning_agent)
graph.add_node("rag", retrieval_agent)
graph.add_node("tool", tool_execution_agent)
graph.add_node("synth", synthesis_agent)

graph.set_entry_point("planner")

graph.add_conditional_edges(
    "planner",
    route_agent,
    {"rag": "rag", "tool": "tool"}
)

graph.add_edge("rag", "synth")
graph.add_edge("tool", "synth")
graph.add_edge("synth", END)

app_graph = graph.compile()
print("✅ LangGraph compiled")


✅ LangGraph compiled


### Task 5: Handle User Query

In [15]:
def handle_user_query(user_query: str) -> Dict[str, Any]:
    init_state: WorkflowState = {
        "user_query": user_query,
        "operation": "",
        "plan": "",
        "react_steps": [],
        "retrieved_context": "",
        "citations": [],
        "tool_name": "",
        "tool_input": {},
        "tool_result": {},
        "final_answer": "",
        "error": "",
    }
    out = app_graph.invoke(init_state)

    return {
        "query": user_query,
        "operation": out.get("operation"),
        "plan": out.get("plan"),
        "answer": out.get("final_answer"),
        "citations": out.get("citations", []),
        "react_steps": out.get("react_steps", []),
        "tool_name": out.get("tool_name", ""),
        "tool_result": out.get("tool_result", {}),
    }



### Task 6: User Interface

In [16]:
import gradio as gr
import json

def interface_function(user_query: str):
    res = handle_user_query(user_query)

    answer = res.get("answer", "")

    citations = res.get("citations", [])
    if citations:
        citation_text = "\n".join(
            [f"- {c['source']} (page {c['page']})" for c in citations]
        )
    else:
        citation_text = "No citations available."

    react_trace = json.dumps(res.get("react_steps", []), indent=2)

    return answer, citation_text, react_trace


with gr.Blocks(title="Multi-Agent RAG + Tools (No API Key)") as demo:
    gr.Markdown("## 🧠 Multi-Agent RAG System")

    with gr.Row():

        # ========== LEFT SIDE ==========
        with gr.Column(scale=2):
            user_query = gr.Textbox(
                label="Ask your question",
                placeholder="Examples: weather in Chennai | calculate (10+20)/2 | Applications of AI",
                lines=3
            )

            ask_btn = gr.Button("Ask")

            answer_box = gr.Textbox(
                label="Answer",
                lines=5,
                interactive=False,
            )

            citation_box = gr.Textbox(
                label="Citations",
                lines=5,
                interactive=False,
            )

        # ========== RIGHT SIDE ==========
        with gr.Column(scale=1):
            react_box = gr.Code(
                label="ReAct Trace (Reason → Act → Observe)",
                language="json",
                lines=24
            )

    ask_btn.click(
        fn=interface_function,
        inputs=user_query,
        outputs=[answer_box, citation_box, react_box]
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
